# 02. Data Cleaning

이 노트북은 `data/raw/`에 저장된 서울 아파트 매매 실거래가 원본 CSV를 불러와 컬럼 구조를 확인하고, 기본 전처리 및 기본 파생 변수를 생성한다.

## 전처리 범위

- 원본 CSV 로드 (헤더 행 자동 탐지)
- 컬럼 구조와 데이터 크기 확인
- 원본 데이터 수집 기간 및 지역 검증
- 분석에 필요한 컬럼 선별
- 컬럼명 영문 snake_case로 정리
- 거래금액, 전용면적, 층, 건축년도 숫자형 변환
- 거래일자, 거래연도, 거래월 생성
- 구, 법정동 분리 및 검증
- 취소 거래 제외
- `-`로 표시된 빈 문자열을 결측치로 변환
- **이상치 처리** (area_m2, price_10k_krw, age 기준 제거) ← 파생 변수 생성 전에 수행
- 기본 파생 변수 생성: 연식, ㎡당 거래가격, 주소 후보
- 전처리 결과를 `data/interim/`에 저장

## 노트북 역할 범위

| 노트북 | 범위 |
| --- | --- |
| `02_data_cleaning` | 원본 CSV 로드, 기본 컬럼 정리, 결측/이상치 처리, 기본 파생 변수 (연식, ㎡당 가격, 주소 후보) |
| `03_feature_engineering` | 외부 데이터 결합, 거리 기반 변수 생성, 추가 파생 변수 생성 |

## 1. 라이브러리 및 경로 설정


In [35]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DATA_PATH = PROJECT_ROOT / 'data/raw/seoul_apt_trade_2025_raw.csv'
INTERIM_DATA_PATH = PROJECT_ROOT / 'data/interim/seoul_apt_trade_2025_basic_cleaned.csv'

RAW_DATA_PATH.exists()

True

## 2. 원본 CSV 불러오기

국토교통부 실거래가 공개시스템에서 내려받은 CSV는 상단에 안내 문구와 검색 조건이 포함되어 있다.
헤더 행 위치는 파일 배포 버전마다 달라질 수 있으므로, `skiprows`를 고정값으로 지정하지 않고
`거래금액`·`전용면적` 키워드를 포함한 행을 자동 탐지하여 동적으로 결정한다.
파일 인코딩은 `cp949`로 읽는다.

In [36]:
def find_header_row(filepath, encoding='cp949', keywords=('거래금액', '전용면적')):
    """실제 컬럼 헤더 행의 인덱스를 자동으로 탐지한다."""
    with open(filepath, encoding=encoding) as f:
        for i, line in enumerate(f):
            if all(kw in line for kw in keywords):
                return i
    raise ValueError(
        f"헤더 행을 찾을 수 없습니다. 파일 형식을 확인하세요. 탐색 키워드: {keywords}"
    )

header_row = find_header_row(RAW_DATA_PATH)
print(f"헤더 행: {header_row + 1}번째 줄에서 탐지 (skiprows={header_row})")

raw_df = pd.read_csv(RAW_DATA_PATH, encoding='cp949', skiprows=header_row)
raw_df.shape

헤더 행: 16번째 줄에서 탐지 (skiprows=15)


(83763, 20)

In [37]:
raw_df.head()

,NO,시군구,번지,본번,부번,단지명,전용면적(㎡),계약년월,계약일,거래금액(만원),동,층,매수자,매도자,건축년도,도로명,해제사유발생일,거래유형,중개사소재지,등기일자
0,1,서울특별시 광진구 구의동,611,611,0,구의현대2단지,84.910,202512,31,"210,000",-,22,개인,개인,1996,광나루로56길 32,20260205,중개거래,서울 광진구,-
1,2,서울특별시 성동구 하왕십리동,1002,1002,0,왕십리KCC스위첸,64.236,202512,31,"137,500",101,8,개인,개인,2016,무학봉길 35,-,중개거래,서울 성동구,26.03.25
2,3,서울특별시 종로구 행촌동,41-1,41,1,대성아파트,94.940,202512,31,"69,500",가동,3,개인,개인,1971,사직로 21,-,중개거래,서울 서초구,26.02.26
3,4,서울특별시 중구 충무로4가,306,306,0,남산센트럴자이,82.413,202512,31,"128,000",남산센트럴 자이,18,개인,개인,2009,퇴계로 235,-,중개거래,서울 중구,26.04.02
4,5,서울특별시 성동구 마장동,818,818,0,현대,84.910,202512,31,"118,000",107,12,개인,개인,1998,살곶이길 50,-,중개거래,서울 성동구,26.02.12


In [38]:
raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 83763 entries, 0 to 83762
Data columns (total 20 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   NO        83763 non-null  int64  
 1   시군구       83763 non-null  str    
 2   번지        83763 non-null  str    
 3   본번        83763 non-null  int64  
 4   부번        83763 non-null  int64  
 5   단지명       83763 non-null  str    
 6   전용면적(㎡)   83763 non-null  float64
 7   계약년월      83763 non-null  int64  
 8   계약일       83763 non-null  int64  
 9   거래금액(만원)  83763 non-null  str    
 10  동         83763 non-null  str    
 11  층         83763 non-null  int64  
 12  매수자       83763 non-null  str    
 13  매도자       83763 non-null  str    
 14  건축년도      83763 non-null  int64  
 15  도로명       83763 non-null  str    
 16  해제사유발생일   83763 non-null  str    
 17  거래유형      83763 non-null  str    
 18  중개사소재지    83763 non-null  str    
 19  등기일자      83763 non-null  str    
dtypes: float64(1), int64(7), str(12)
memory

In [39]:
# 수집 조건 검증: 서울특별시 2025년 데이터만 포함되어야 한다.
years = raw_df['계약년월'].dropna().astype(str).str[:4].unique()
assert (years == '2025').all(), f"2025년 외 연도 포함: {years[years != '2025']}"

assert raw_df['시군구'].dropna().str.startswith('서울특별시').all(), \
    "서울특별시 외 지역 데이터가 포함되어 있습니다."

print(f"✓ 기간 검증 통과: 계약년도 {sorted(years)}")
print(f"✓ 지역 검증 통과: 전체 {len(raw_df):,}건 모두 서울특별시")

✓ 기간 검증 통과: 계약년도 ['2025']
✓ 지역 검증 통과: 전체 83,763건 모두 서울특별시


## 3. 필요한 컬럼 선별 및 컬럼명 정리


In [40]:
selected_columns = [
    '시군구',
    '단지명',
    '전용면적(㎡)',
    '계약년월',
    '계약일',
    '거래금액(만원)',
    '층',
    '건축년도',
    '도로명',
]

cancelled_mask = raw_df['해제사유발생일'].notna() & (raw_df['해제사유발생일'].astype(str).str.strip() != '-')
print(f'전체 거래 수: {len(raw_df):,}')
print(f'취소 거래 수: {cancelled_mask.sum():,}')
print(f'취소 거래 제외 후 거래 수: {(~cancelled_mask).sum():,}')

df = raw_df.loc[~cancelled_mask, selected_columns].copy()

df = df.rename(columns={
    '시군구': 'sigungu',
    '단지명': 'apartment_name',
    '전용면적(㎡)': 'area_m2',
    '계약년월': 'contract_ym',
    '계약일': 'contract_day',
    '거래금액(만원)': 'price_10k_krw',
    '층': 'floor',
    '건축년도': 'built_year',
    '도로명': 'road_name',
})

df.head()

전체 거래 수: 83,763
취소 거래 수: 6,404
취소 거래 제외 후 거래 수: 77,359


,sigungu,apartment_name,area_m2,contract_ym,contract_day,price_10k_krw,floor,built_year,road_name
1,서울특별시 성동구 하왕십리동,왕십리KCC스위첸,64.236,202512,31,"137,500",8,2016,무학봉길 35
2,서울특별시 종로구 행촌동,대성아파트,94.940,202512,31,"69,500",3,1971,사직로 21
3,서울특별시 중구 충무로4가,남산센트럴자이,82.413,202512,31,"128,000",18,2009,퇴계로 235
4,서울특별시 성동구 마장동,현대,84.910,202512,31,"118,000",12,1998,살곶이길 50
5,서울특별시 중구 충무로4가,남산센트럴자이,80.473,202512,31,"125,000",8,2009,퇴계로 235


## 4. 빈 값 표기 및 기본 타입 변환

원본 데이터에서 `-`는 값이 없음을 나타내는 표기로 사용된다. 분석에 사용할 문자열 컬럼에서는 `-`를 결측치로 변환하고, 숫자형 컬럼은 `pd.to_numeric(..., errors='coerce')`를 통해 변환 불가능한 값을 결측치로 처리한다.


In [41]:
df['price_10k_krw'] = (
    df['price_10k_krw']
    .astype(str)
    .str.replace(',', '', regex=False)
    .pipe(pd.to_numeric, errors='coerce')
)

string_columns = ['sigungu', 'apartment_name', 'road_name']
for column in string_columns:
    df[column] = (
        df[column]
        .astype('string')
        .str.strip()
        .replace({'': pd.NA, '-': pd.NA})
    )

numeric_columns = ['area_m2', 'contract_ym', 'contract_day', 'floor', 'built_year']
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')

df.dtypes


sigungu               str
apartment_name        str
area_m2           float64
contract_ym         int64
contract_day        int64
price_10k_krw       int64
floor               int64
built_year          int64
road_name             str
dtype: object

## 5. 지역 및 거래일자 변수 생성


In [42]:
location_parts = df['sigungu'].str.split(expand=True)
df['sido'] = location_parts[0]
df['gu'] = location_parts[1]
df['law_dong'] = location_parts.iloc[:, 2:].apply(lambda row: ' '.join(row.dropna()), axis=1)

df['contract_year'] = df['contract_ym'] // 100
df['contract_month'] = df['contract_ym'] % 100
df['contract_date'] = pd.to_datetime(
    df['contract_year'].astype('Int64').astype(str) + '-' +
    df['contract_month'].astype('Int64').astype(str).str.zfill(2) + '-' +
    df['contract_day'].astype('Int64').astype(str).str.zfill(2),
    errors='coerce'
)

df[['sigungu', 'sido', 'gu', 'law_dong', 'contract_date']].head()

,sigungu,sido,gu,law_dong,contract_date
1,서울특별시 성동구 하왕십리동,서울특별시,성동구,하왕십리동,2025-12-31
2,서울특별시 종로구 행촌동,서울특별시,종로구,행촌동,2025-12-31
3,서울특별시 중구 충무로4가,서울특별시,중구,충무로4가,2025-12-31
4,서울특별시 성동구 마장동,서울특별시,성동구,마장동,2025-12-31
5,서울특별시 중구 충무로4가,서울특별시,중구,충무로4가,2025-12-31


In [43]:
# sigungu 분리 결과 검증
assert df['sido'].eq('서울특별시').all(), \
    "sido 컬럼에 '서울특별시' 외 값이 존재합니다."
assert df['gu'].dropna().str.endswith('구').all(), \
    "gu 컬럼 추출 이상: '구'로 끝나지 않는 값이 존재합니다."

print("✓ sigungu 분리 검증 통과")
print(f"  sido 고유값: {df['sido'].unique()}")
print(f"  gu 고유값 수: {df['gu'].nunique()}개")

✓ sigungu 분리 검증 통과
  sido 고유값: <StringArray>
['서울특별시']
Length: 1, dtype: str
  gu 고유값 수: 25개


## 6. 도로명 주소 보완

원본 데이터에서 일부 거래 행의 `도로명`이 공백으로 제공된다. 같은 `sigungu`와 `apartment_name`을 가진 다른 거래 행에 도로명이 있으면 최빈값으로 보완하고, 그래도 찾지 못하면 결측치로 유지한다.


In [ ]:
missing_road_before = df['road_name'].isna().sum()

road_name_fill_map = (
    df.dropna(subset=['road_name'])
    .groupby(['sigungu', 'apartment_name'])['road_name']
    .agg(lambda values: values.mode().iloc[0] if not values.mode().empty else values.iloc[0])
)

missing_road_mask = df['road_name'].isna()
road_name_candidates = df.loc[missing_road_mask, ['sigungu', 'apartment_name']].apply(
    lambda row: road_name_fill_map.get((row['sigungu'], row['apartment_name']), pd.NA),
    axis=1,
)

df.loc[missing_road_mask, 'road_name'] = road_name_candidates.values
missing_road_after = df['road_name'].isna().sum()

print(f'도로명 결측 보완 전: {missing_road_before:,}행')
print(f'같은 아파트명 기준 보완: {missing_road_before - missing_road_after:,}행')
print(f'보완 후에도 도로명 결측: {missing_road_after:,}행')

df.loc[df['road_name'].isna(), ['sigungu', 'apartment_name', 'built_year']].drop_duplicates()


## 7. 취소 거래 처리 결과 확인

`해제사유발생일`은 최종 분석 변수로 사용하지 않지만, 취소 거래를 제외하기 위해 원본 데이터에서만 임시로 활용했다.


In [44]:
pd.Series({
    'raw_rows': len(raw_df),
    'cancelled_rows': int(cancelled_mask.sum()),
    'cleaned_rows': len(df),
})

raw_rows          83763
cancelled_rows     6404
cleaned_rows      77359
dtype: int64

In [45]:
df.shape

(77359, 15)

## 8. 이상치 처리

분석에 사용할 수 없는 이상치 행을 제거한다.
`age`는 이 단계에서 먼저 계산하여 음수 여부를 확인한 뒤, `price_per_m2_10k_krw` 생성 전에 제거를 완료한다.

| 조건 | 처리 방침 | 이유 |
| --- | --- | --- |
| `area_m2 <= 0` | 제거 | 유효하지 않은 면적 |
| `price_10k_krw <= 0` | 제거 | 유효하지 않은 거래금액 |
| `contract_date` 결측 | 제거 | 날짜 없이 시계열 분석 불가 |
| `gu` 결측 | 제거 | 지역 변수 없이 분석 불가 |
| `law_dong` 결측 | 제거 | 지역 변수 없이 분석 불가 |
| `age < 0` | 건수 확인 후 제거 | 건축년도 오기재 가능성 높음. 신축·입주예정 케이스 여부 팀 확인 필요 |

In [46]:
# age 먼저 계산 (이상치 판별에 필요)
df['age'] = df['contract_year'] - df['built_year']

# age < 0: 건수와 내용을 먼저 출력한 뒤 제거
age_anomaly = df[df['age'] < 0]
if len(age_anomaly) > 0:
    print(f"[age < 0] 제거 대상 {len(age_anomaly):,}건 — 원본 건축년도 확인 필요:")
    print(age_anomaly[['apartment_name', 'gu', 'built_year', 'contract_year', 'age']].to_string())
else:
    print("[age < 0] 해당 없음")

before_count = len(df)

df = df[
    (df['area_m2'] > 0) &
    (df['price_10k_krw'] > 0) &
    (df['contract_date'].notna()) &
    (df['gu'].notna()) &
    (df['law_dong'].notna()) &
    (df['age'] >= 0)
].copy()

after_count = len(df)
print(f"\n제거 전: {before_count:,}행 → 제거 후: {after_count:,}행 (제거: {before_count - after_count:,}행)")

[age < 0] 해당 없음

제거 전: 77,359행 → 제거 후: 77,359행 (제거: 0행)


## 9. 기본 파생 변수 생성

이상치 제거 완료 후 `price_per_m2_10k_krw`와 `full_road_address`를 생성한다.
`area_m2 <= 0` 행이 제거된 이후이므로 나눗셈 결과에 `inf`나 음수가 발생하지 않는다.

In [47]:
df['price_per_m2_10k_krw'] = df['price_10k_krw'] / df['area_m2']
df['full_road_address'] = pd.NA
valid_road_mask = df['road_name'].notna()
df.loc[valid_road_mask, 'full_road_address'] = (
    df.loc[valid_road_mask, 'sido'] + ' ' +
    df.loc[valid_road_mask, 'gu'] + ' ' +
    df.loc[valid_road_mask, 'road_name']
)

df[[
    'apartment_name', 'gu', 'law_dong', 'contract_date', 'area_m2',
    'floor', 'built_year', 'age', 'price_10k_krw', 'price_per_m2_10k_krw',
    'road_name', 'full_road_address'
]].head()


,apartment_name,gu,law_dong,contract_date,area_m2,floor,built_year,age,price_10k_krw,price_per_m2_10k_krw
1,왕십리KCC스위첸,성동구,하왕십리동,2025-12-31,64.236,8,2016,9,137500,2140.544243
2,대성아파트,종로구,행촌동,2025-12-31,94.940,3,1971,54,69500,732.041289
3,남산센트럴자이,중구,충무로4가,2025-12-31,82.413,18,2009,16,128000,1553.153022
4,현대,성동구,마장동,2025-12-31,84.910,12,1998,27,118000,1389.706748
5,남산센트럴자이,중구,충무로4가,2025-12-31,80.473,8,2009,16,125000,1553.316019


## 10. 결측치 최종 확인

In [48]:
df.isna().sum().sort_values(ascending=False)

sigungu                 0
apartment_name          0
price_per_m2_10k_krw    0
age                     0
contract_date           0
contract_month          0
contract_year           0
law_dong                0
gu                      0
sido                    0
road_name               0
built_year              0
floor                   0
price_10k_krw           0
contract_day            0
contract_ym             0
area_m2                 0
full_road_address       0
dtype: int64

In [49]:
df[[
    'price_10k_krw', 'price_per_m2_10k_krw', 'area_m2',
    'floor', 'built_year', 'age'
]].describe()

,price_10k_krw,price_per_m2_10k_krw,area_m2,floor,built_year,age
count,7.735900e+04,77359.000000,77359.000000,77359.000000,77359.000000,77359.000000
mean,1.269145e+05,1640.967786,76.074963,9.712496,2003.318528,21.681472
std,9.483078e+04,890.164494,27.793934,6.479812,11.402045,11.402045
min,6.500000e+03,181.582361,11.330000,-2.000000,1961.000000,0.000000
25%,7.050000e+04,1033.525018,59.785000,5.000000,1996.000000,13.000000
50%,1.045000e+05,1412.835136,81.750000,9.000000,2003.000000,22.000000
75%,1.540000e+05,1971.253730,84.960000,13.000000,2012.000000,29.000000
max,2.900000e+06,10586.723519,317.360000,67.000000,2025.000000,64.000000


In [50]:
# 이상치 제거 결과 검증
remaining = df.query('area_m2 <= 0 or price_10k_krw <= 0 or age < 0')
assert len(remaining) == 0, f"이상치 {len(remaining):,}건이 남아있습니다."
print("✓ 이상치 제거 검증 통과")
print(f"최종 데이터: {df.shape[0]:,}행 × {df.shape[1]}열")

✓ 이상치 제거 검증 통과
최종 데이터: 77,359행 × 18열


## 11. 전처리 결과 저장

기본 전처리 결과는 중간 산출물이므로 `data/interim/`에 저장한다. 해당 폴더의 실제 CSV 파일은 `.gitignore`에 의해 GitHub에 업로드하지 않는다.

In [51]:
INTERIM_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(INTERIM_DATA_PATH, index=False, encoding='utf-8-sig')
INTERIM_DATA_PATH

PosixPath('/Users/cheong-kyumin/HSU/데이터마이닝/dm-apt-predict/dm-apt-predict/data/interim/seoul_apt_trade_2025_basic_cleaned.csv')